# 03 — UniGaze verification, baselines and calibration

**Purpose**: establish how good *simple* methods are before any fine-tuning, so later gains are measured honestly.

| Baseline | What it is | Why |
|---|---|---|
| B1 | screen centre; mean training target | floor: a model must beat "always predict the middle" |
| B2a | ridge regression on frozen UniGaze CLS features (768-d) | does pretrained gaze knowledge already encode screen position? |
| B2b | ridge on UniGaze gaze angles + head pose + face/eye position (≈14-d), linear and quadratic | the "frozen gaze output → screen mapping" route |
| B3 | small MLP head on frozen backbone | trained in notebook 04, Stage A (primary deep baseline) |
| B4 | last blocks fine-tuned | notebook 04, Stage B |

Ridge strength is chosen by **leave-one-participant-out cross-validation on the six training participants**;
validation participants are used only to score. The test set is not touched.

**Label compatibility (Approach A vs B)**: UniGaze outputs gaze *direction* (pitch, yaw). Your labels are screen
pixels. Converting pixels to 3D directions needs camera intrinsics, the camera's pose relative to the monitor and
the monitor's physical size — none were recorded (see `reports/dataset_audit.json → geometry_available`).
Fabricating them would create wrong angular labels, so **Approach B** is used: screen coordinates are regressed
directly and UniGaze's angles serve as inputs.

**GPU**: feature extraction (§4) runs the 86 M-parameter backbone over 40,000 images — GPU recommended
(roughly 3–8 minutes on an RTX 5060 laptop GPU; CPU is far slower).

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

In [ ]:
import json, numpy as np, pandas as pd, torch
from src.dataset import require_audit_pass
from src.utils import select_device, seed_everything, environment_report, new_run_dir, write_json
require_audit_pass(cfg)
seed_everything(cfg["reproducibility"]["seed"])
hw = cfg["hardware"]
dev = select_device(hw["device"], allow_cpu=hw["allow_cpu_training"], precision=hw["precision"])
print(dev)

## 1. Licence notice
UniGaze weights are released under the **ModelGo Attribution-NonCommercial-ResponsibleAI License 2.0
(MG-NC-RAI-2.0)**. Use for this NASA Space Apps accessibility project (non-commercial research/education) is the
intended kind of use, but any model you fine-tune from them is a derivative under the same terms: no commercial use,
attribution required, responsible-AI restrictions apply. Read `LICENSE_NOTICE_UNIGAZE.md` and the full licence before
any redistribution, hosting as a service, or commercial use.

## 2. Download the checkpoint — opt-in (≈350 MB, once)
Downloads `UniGaze/UniGaze-models/unigaze_b16_joint.safetensors` from Hugging Face into
`checkpoints/hf_cache/`. Nothing is downloaded unless you set `ALLOW_DOWNLOAD = True`.

In [ ]:
ALLOW_DOWNLOAD = False             # <- set True once, deliberately
from src.models import resolve_checkpoint
try:
    ckpt_path = resolve_checkpoint(cfg, allow_download=ALLOW_DOWNLOAD)
    print("checkpoint:", ckpt_path, f"({ckpt_path.stat().st_size / 1e6:.0f} MB)")
except FileNotFoundError as e:
    print(e)

## 3. Load and verify the pretrained model
Checks: strict key match (no missing/unexpected weights), 86,569,194 parameters (ViT-B/16 + gaze head), output
shape (B, 2) = (pitch, yaw) in radians, and plausible angles on real normalised validation faces (typically within
±0.6 rad). The SHA-256 is recorded in every experiment.

In [ ]:
from src.models import load_unigaze, count_parameters
from src.pipeline import split_source
from src.preprocessing import to_model_tensor
unigaze, ckpt_info = load_unigaze(cfg, allow_download=False)
print(json.dumps(ckpt_info, indent=2))
assert ckpt_info["parameters"] == 86_569_194, "unexpected architecture"
va_df, va_src, va_geo = split_source(cfg, "validation")
idx = np.where(va_src.ok)[0][:16]
x = to_model_tensor(np.stack([va_src(i) for i in idx])).to(dev.device)
with torch.no_grad():
    g = unigaze.to(dev.device).eval()(x)["pred_gaze"].float().cpu().numpy()
print("pitch/yaw (deg) on 16 validation faces:\n", np.degrees(g).round(1))

## 4. Extract frozen features — GPU
Runs the frozen backbone once per frame (train + validation) and caches CLS features and gaze angles under
`processed_cache/` keyed by checkpoint hash and preprocessing version. Reruns load from disk.

In [ ]:
RUN_FEATURE_EXTRACTION = False     # <- set True (GPU, a few minutes)
from src.pipeline import extract_split_features
if RUN_FEATURE_EXTRACTION:
    feats = {s: extract_split_features(cfg, unigaze, ckpt_info["sha256"], s, dev.device, dev.precision)
             for s in ("train", "validation")}
    for s, b in feats.items():
        print(s, tuple(b["features"].shape), "usable:", int(np.sum(b["usable"])))

## 5. B1 — naive predictors
These numbers are the bar every learned model must clear by a wide margin.

In [ ]:
from src.pipeline import split_source
from src.evaluation import baseline_predictions, prediction_table, evaluate_predictions
tr_df, tr_src, tr_geo = split_source(cfg, "train")
va_df, va_src, va_geo = split_source(cfg, "validation")
tu, vu = np.asarray(tr_src.ok), np.asarray(va_src.ok)
results, pred_tables = {}, {}
for kind in ("center", "train_mean"):
    p = baseline_predictions(tr_df[tu], int(vu.sum()), kind)
    pt = prediction_table(va_df[vu], p, W, H, f"baseline_{kind}", cfg["preprocessing"]["version"], kind)
    results[f"B1_{kind}"] = evaluate_predictions(pt, cfg, bootstrap=False)
    pred_tables[f"B1_{kind}"] = pt
    print(kind, round(results[f"B1_{kind}"]["frame_level"]["mean_euclid_px"], 1), "px")

## 6. B2a / B2b — ridge regression on frozen UniGaze outputs
`fit_ridge_grouped` standardises features and picks the ridge strength by leave-one-**participant**-out CV
over the training participants only. Quadratic terms (B2b-poly2) let the mapping bend near screen edges.
**Interpretation**: if B2b ≈ B2a, the low-dimensional gaze/head signal carries most of the information; if B2a is
much better, the appearance features hold extra information a learned head can exploit.

In [ ]:
from src.baselines import fit_ridge_grouped, geometric_features
from src.pipeline import extract_split_features, features_path
missing = [s for s in ("train", "validation") if not features_path(cfg, s, ckpt_info["sha256"]).is_file()]
if missing and not RUN_FEATURE_EXTRACTION:
    raise RuntimeError(f"Features for {missing} not extracted yet - set RUN_FEATURE_EXTRACTION = True in section 4.")
ftr = extract_split_features(cfg, unigaze, ckpt_info["sha256"], "train", dev.device, dev.precision)
fva = extract_split_features(cfg, unigaze, ckpt_info["sha256"], "validation", dev.device, dev.precision)
Ytr = tr_df[["target_x_norm", "target_y_norm"]].to_numpy()[tu]
groups = tr_df["participant_id"].to_numpy()[tu]

def score(name, model, Xva):
    p = model.predict(Xva)
    pt = prediction_table(va_df[vu], p, W, H, name, cfg["preprocessing"]["version"], name)
    results[name] = evaluate_predictions(pt, cfg, bootstrap=False); pred_tables[name] = pt
    print(f"{name:18s} val mean {results[name]['frame_level']['mean_euclid_px']:.1f}px | "
          f"median {results[name]['frame_level']['median_euclid_px']:.1f}px")

r = fit_ridge_grouped(ftr["features"].numpy()[tu], Ytr, groups, width=W, height=H)
print("B2a alpha", r["alpha"], r["cv_mean_euclid_px_by_alpha"]); score("B2a_ridge_cls", r["model"], fva["features"].numpy()[vu])
Gtr = geometric_features(ftr["gaze_pitchyaw"].numpy()[tu], tr_geo[tu], tr_df[tu])
Gva = geometric_features(fva["gaze_pitchyaw"].numpy()[vu], va_geo[vu], va_df[vu])
for deg in (1, 2):
    r = fit_ridge_grouped(Gtr, Ytr, groups, poly_degree=deg, width=W, height=H)
    print(f"B2b poly{deg} alpha", r["alpha"]); score(f"B2b_geom_poly{deg}", r["model"], Gva)

## 7. Comparison table (validation participants P007, P008)

In [ ]:
rows = []
for name, res in results.items():
    f, t = res["frame_level"], res["trial_level"]
    pp = {d["participant_id"]: d["mean_euclid_px"] for d in res["per_participant"]}
    rows.append({"model": name, "mean_px": f["mean_euclid_px"], "median_px": f["median_euclid_px"],
                 "p95_px": f["p95_euclid_px"], "trial_mean_px": t["mean_euclid_px"],
                 "pct_diag": f["mean_euclid_pct_diag"], "within_100px": f["within_100px"], **pp})
baseline_table = pd.DataFrame(rows).sort_values("mean_px")
run = new_run_dir(cfg.paths.experiments, "baselines")
baseline_table.to_csv(run / "baseline_comparison.csv", index=False)
write_json(run / "results.json", {"results": results, "unigaze_checkpoint": ckpt_info,
                                  "environment": environment_report()})
for k, pt in pred_tables.items():
    pt.to_parquet(run / f"val_predictions_{k}.parquet", index=False)
print("saved to", run)
baseline_table.round(3)

## 8. Personalised calibration on the validation participants
Protocol (no frame used for both fitting and scoring): calibration points come from session **S01** only
(the S01 trial nearest to each canonical layout point); scoring uses sessions **S02–S05**; the uncalibrated row
(k = 0) is scored on exactly the same frames. Calibration hyper-parameters (method, ridge strength) may be chosen
here because these are validation participants.

In [ ]:
from src.calibration import evaluate_calibration
best_name = baseline_table.iloc[0]["model"]
cal_rows = []
for method in ("ridge_affine", "ridge_poly2"):
    for alpha in (0.01, 0.05, 0.2, 1.0):
        res = evaluate_calibration(pred_tables[best_name], cfg, (5, 9, 16), method, alpha)
        res["alpha"] = alpha; res["requested_method"] = method
        cal_rows.append(res)
cal = pd.concat(cal_rows)
cal.to_csv(run / f"calibration_{best_name}.csv", index=False)
cal.groupby(["requested_method", "alpha", "k_points"])["mean_euclid_px"].mean().unstack().round(1)

In [ ]:
from src.visualization import plot_comparison
plot_comparison(baseline_table, "model", "mean_px", run / "figures" / "baseline_comparison",
                "Baselines on validation participants")
best_cal = cal[(cal.requested_method == cfg["calibration"]["method"]) & (cal.alpha == cfg["calibration"]["ridge_alpha"])]
plot_comparison(best_cal, "k_points", "mean_euclid_px", run / "figures" / "calibration_vs_points",
                f"Calibration of {best_name} (k=0: none)", hue="participant_id", kind="line")
print("figures in", run / "figures")